In [21]:
import pandas as pd
import glob
import os

## **gold_raw**

### **Data Understanding**

In [ ]:
# 1. กำหนด Path ไปยังไฟล์ดิบ
input_path = '../../data/raw/gold_raw'
all_files = glob.glob(os.path.join(input_path, "gold_raw_uncleaned_*.csv"))

print(f"🔎 ตรวจพบไฟล์ดิบทั้งหมด {len(all_files)} ไฟล์\n")

for file in sorted(all_files):
    print(f"{'='*50}")
    print(f"📊 ไฟล์: {os.path.basename(file)}")
    
    # อ่านข้อมูลแบบ String ทั้งหมดเพื่อดูสภาพดั้งเดิม
    df_raw = pd.read_csv(file, dtype=str)
    
    # ดู 5 แถวแรกเพื่อดูหัวตารางและรูปแบบข้อมูล
    print("\n[5 แถวแรก]")
    print(df_raw.head())
    
    # ดูโครงสร้างและค่าว่าง
    print("\n[สรุปเบื้องต้น]")
    print(f"- จำนวนแถวทั้งหมด: {len(df_raw)}")
    print(f"- จำนวนคอลัมน์: {len(df_raw.columns)}")
    
    # ตรวจสอบว่ามีคอลัมน์ไหนบ้างที่มีค่าว่างเยอะผิดปกติ
    missing = df_raw.isnull().sum()
    if missing.any():
        print("\n[ค่าว่างที่พบ]")
        print(missing[missing > 0])
    
    # สุ่มดูแถวที่มีแนวโน้มจะเป็น "แถวขยะ" (เช่น แถวที่มีคำว่า 'ลำดับ' ซ้ำ)
    col_round_name = df_raw.columns[1]
    
    # เช็คว่าค่าในคอลัมน์ Round 'ไม่' เป็นตัวเลข
    is_numeric = pd.to_numeric(df_raw[col_round_name], errors='coerce').notnull()
    non_numeric_rows = df_raw[~is_numeric]
    
    if not non_numeric_rows.empty:
        print(f"\n[ตัวอย่างแถวที่อาจเป็นขยะ (ในคอลัมน์ {col_round_name})]")
        print(non_numeric_rows.head(3))

print(f"\n{'='*50}")
print("✅ สำรวจข้อมูลเบื้องต้นเสร็จสิ้น")

🔎 ตรวจพบไฟล์ดิบทั้งหมด 6 ไฟล์

📊 ไฟล์: gold_raw_uncleaned_2564.csv

[5 แถวแรก]
                                                   0  \
0                                                NaN   
1                                        วันที่/เวลา   
2  คำแนะนำ : « เลื่อนหน้าจอซ้ายขวาเพื่อดูข้อมูลเพ...   
3  เสาร์ที่ 30 มกราคม 64-5030/01/2564 09:25126,10...   
4                                   30/01/2564 09:25   

                                                   1  \
0                                                NaN   
1                                           ครั้งที่   
2  คำแนะนำ : « เลื่อนหน้าจอซ้ายขวาเพื่อดูข้อมูลเพ...   
3  เสาร์ที่ 30 มกราคม 64-5030/01/2564 09:25126,10...   
4                                                  1   

                                                   2  \
0                                            ทองแท่ง   
1                                       รับซื้อ(บาท)   
2  คำแนะนำ : « เลื่อนหน้าจอซ้ายขวาเพื่อดูข้อมูลเพ...   
3  เสาร์ที่ 30 มกราคม 6

### **Data Preparation**

In [ ]:
# 1. Setup Path (ปรับตามโครงสร้างโฟลเดอร์ที่วางไว้)
input_path = '../../data/raw/gold_raw'
output_path = '../../data/processed/gold_all_years_final.csv'

# กำหนดหัวตารางและคอลัมน์ตัวเลข
clean_headers = [
    'Datetime', 'Round', 'GoldBar_Buy', 'GoldBar_Sell',
    'Ornament_Base', 'Ornament_Sell', 'Gold_Spot', 
    'THB_Rate', 'Change', 'Month', 'Year'
]
numeric_cols = ['GoldBar_Buy', 'GoldBar_Sell', 'Ornament_Base', 'Ornament_Sell', 'Gold_Spot', 'THB_Rate', 'Change']

# 2. กระบวนการ Clean before Merge
all_files = glob.glob(os.path.join(input_path, "gold_raw_uncleaned_*.csv"))
cleaned_list = []

for file_name in sorted(all_files):
    df = pd.read_csv(file_name, dtype=str)
    
    # คลีนแถวขยะ (เช็คคอลัมน์ Round)
    df.iloc[:, 1] = pd.to_numeric(df.iloc[:, 1], errors='coerce')
    df_clean = df.dropna(subset=[df.columns[1]]).copy()
    
    # สวมมงกุฎหัวตาราง
    df_clean.columns = clean_headers
    
    # ลบ Comma และแปลงเป็น Numeric
    for col in numeric_cols:
        df_clean[col] = df_clean[col].astype(str).str.replace(',', '', regex=False)
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    cleaned_list.append(df_clean)

# 3. รวมเป็นไฟล์เดียว
df_final = pd.concat(cleaned_list, ignore_index=True)
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ คลีนและรวมไฟล์เสร็จสมบูรณ์! ข้อมูลทั้งหมด: {len(df_final)} แถว")

✅ คลีนและรวมไฟล์เสร็จสมบูรณ์! ข้อมูลทั้งหมด: 9298 แถว


In [24]:
df_gold = pd.read_csv('../../data/processed/gold_all_years_final.csv')
df_gold

,Datetime,Round,GoldBar_Buy,GoldBar_Sell,Ornament_Base,Ornament_Sell,Gold_Spot,THB_Rate,Change,Month,Year
0,30/01/2564 09:25,1.0,26100.0,26200.0,25635.56,26700.0,1847.5,29.94,50.0,มกราคม,2564
1,29/01/2564 16:05,4.0,26150.0,26250.0,25681.04,26750.0,1850.5,29.95,50.0,มกราคม,2564
2,29/01/2564 14:15,3.0,26100.0,26200.0,25635.56,26700.0,1845.0,29.99,50.0,มกราคม,2564
3,29/01/2564 13:54,2.0,26150.0,26250.0,25681.04,26750.0,1850.0,29.99,50.0,มกราคม,2564
4,29/01/2564 09:26,1.0,26100.0,26200.0,25635.56,26700.0,1843.0,30.00,0.0,มกราคม,2564
...,...,...,...,...,...,...,...,...,...,...,...
9293,02/02/2569 10:32,5.0,71400.0,71600.0,69978.56,72400.0,4790.0,31.59,250.0,กุมภาพันธ์,2569
9294,02/02/2569 10:20,4.0,71650.0,71850.0,70221.12,72650.0,4805.0,31.60,100.0,กุมภาพันธ์,2569
9295,02/02/2569 10:16,3.0,71750.0,71950.0,70312.08,72750.0,4812.0,31.60,150.0,กุมภาพันธ์,2569
9296,02/02/2569 09:51,2.0,71900.0,72100.0,70463.68,72900.0,4823.0,31.60,450.0,กุมภาพันธ์,2569


## **oil_raw**

### **Data Understanding**

In [ ]:
# 1. กำหนด Path (ปรับให้ตรงกับโฟลเดอร์ใน VS Code ของคุณ)
input_path = '../../data/raw/oil_raw' # หรือระบุชื่อไฟล์โดยตรงถ้าอยู่ในโฟลเดอร์เดียวกัน
all_oil_files = glob.glob(os.path.join(input_path, "oil_wti_raw_*.csv"))

print(f"🛢️ ตรวจพบไฟล์น้ำมันดิบทั้งหมด {len(all_oil_files)} ไฟล์\n")

for file in sorted(all_oil_files):
    print(f"{'='*60}")
    print(f"📊 ไฟล์: {os.path.basename(file)}")
    
    # อ่านไฟล์แบบ String เพื่อดูค่าดั้งเดิม
    df_oil_raw = pd.read_csv(file, dtype=str)
    
    # 2. ดูหน้าตาข้อมูล 5 แถวแรก
    print("\n[5 แถวแรก]")
    print(df_oil_raw.head())
    
    # 3. สรุปโครงสร้าง
    print("\n[สรุปข้อมูลเบื้องต้น]")
    print(f"- จำนวนแถว: {df_oil_raw.shape[0]}")
    print(f"- คอลัมน์ที่พบ: {list(df_oil_raw.columns)}")
    
    # 4. ตรวจสอบค่าว่าง (NaN) ในราคา Close
    # ข้อมูลจาก Yahoo Finance มักจะมี NaN ในวันที่ตลาดปิด
    if 'Close' in df_oil_raw.columns:
        nan_count = df_oil_raw['Close'].isnull().sum()
        if nan_count > 0:
            print(f"⚠️ พบค่าว่างในคอลัมน์ Close: {nan_count} แถว (อาจเป็นวันหยุดตลาด)")
            
            # ส่องดูแถวที่มีค่าว่าง (แบบปลอดภัย)
            print("\n[ตัวอย่างแถวที่มีค่าว่าง]:")
            print(df_oil_raw[df_oil_raw['Close'].isnull()].head(3))
        else:
            print("✅ ไม่พบค่าว่างในคอลัมน์ราคา Close")

print(f"\n{'='*60}\n✅ สำรวจข้อมูลน้ำมันดิบเสร็จเรียบร้อย!")

🛢️ ตรวจพบไฟล์น้ำมันดิบทั้งหมด 6 ไฟล์

📊 ไฟล์: oil_wti_raw_2564.csv

[5 แถวแรก]
         Date               Close                High                Low  \
0         NaN                CL=F                CL=F               CL=F   
1  2021-01-04  47.619998931884766   49.83000183105469  47.18000030517578   
2  2021-01-05   49.93000030517578   50.20000076293945   47.2400016784668   
3  2021-01-06  50.630001068115234  50.939998626708984  49.47999954223633   
4  2021-01-07   50.83000183105469  51.279998779296875  50.38999938964844   

                 Open  Volume Year_Scraped  
0                CL=F    CL=F          NaN  
1  48.400001525878906  528525         2564  
2  47.380001068115234  643191         2564  
3   49.81999969482422  509365         2564  
4  50.529998779296875  369292         2564  

[สรุปข้อมูลเบื้องต้น]
- จำนวนแถว: 252
- คอลัมน์ที่พบ: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Year_Scraped']
✅ ไม่พบค่าว่างในคอลัมน์ราคา Close
📊 ไฟล์: oil_wti_raw_2565.csv

[5 แถวแร

### **Data Preparation**

In [25]:
# 1. Setup Path
input_path = '../../data/raw/oil_raw'
output_dir = '../../data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

all_oil_files = glob.glob(os.path.join(input_path, "oil_wti_raw_*.csv"))
cleaned_oil_list = []

print("🚀 เริ่มกระบวนการคลีนข้อมูลน้ำมันดิบ...")

for filename in sorted(all_oil_files):
    year_label = filename.split('_')[-1].replace('.csv', '')
    print(f"กำลังคลีนข้อมูลปี {year_label}...", end=" ")
    
    try:
        # ✨ แก้ไขจุดที่ 1: ใช้ skiprows=[1] เพื่อเอาแถวหัวตารางขยะ (CL=F) ออก
        df = pd.read_csv(filename, skiprows=[1])
        
        # ตรวจสอบว่าคอลัมน์ที่จำเป็นมีอยู่จริง
        required_cols = ['Date', 'Close', 'Year_Scraped']
        df_temp = df[required_cols].copy()
        
        # 1. จัดการค่าว่าง (NaN) 
        # ✨ แก้ไขจุดที่ 2: ใช้ .ffill() แทน method='ffill' เพื่อรองรับ Pandas รุ่นใหม่
        df_temp['Close'] = pd.to_numeric(df_temp['Close'], errors='coerce') # แปลงเป็นตัวเลขก่อน ffill
        df_temp['Close'] = df_temp['Close'].ffill()
        
        # 2. แปลง Date เป็น Datetime Objects
        df_temp['Date'] = pd.to_datetime(df_temp['Date'])
        
        # 3. ลบแถวที่ยังเป็น NaN อยู่ (เช่น วันแรกของปี)
        df_temp = df_temp.dropna(subset=['Close'])
        
        # เปลี่ยนชื่อคอลัมน์ให้ชัดเจน
        df_temp.columns = ['Date', 'Oil_WTI_Close', 'Year_Scraped']
        
        cleaned_oil_list.append(df_temp)
        print(f"✅ สำเร็จ ({len(df_temp)} แถว)")
        
    except Exception as e:
        print(f"❌ Error ในไฟล์ {filename}: {e}")

# 2. รวมไฟล์น้ำมันดิบทั้งหมด
if cleaned_oil_list:
    df_oil_final = pd.concat(cleaned_oil_list, ignore_index=True)
    df_oil_final = df_oil_final.sort_values('Date')
    
    # Save ไฟล์รวม
    final_oil_path = os.path.join(output_dir, 'oil_wti_all_years_cleaned.csv')
    df_oil_final.to_csv(final_oil_path, index=False)
    print(f"\n🎉 คลีนน้ำมันเสร็จเรียบร้อย! บันทึกไปที่: {final_oil_path}")

🚀 เริ่มกระบวนการคลีนข้อมูลน้ำมันดิบ...
กำลังคลีนข้อมูลปี 2564... ✅ สำเร็จ (251 แถว)
กำลังคลีนข้อมูลปี 2565... ✅ สำเร็จ (251 แถว)
กำลังคลีนข้อมูลปี 2566... ✅ สำเร็จ (250 แถว)
กำลังคลีนข้อมูลปี 2567... ✅ สำเร็จ (251 แถว)
กำลังคลีนข้อมูลปี 2568... ✅ สำเร็จ (251 แถว)
กำลังคลีนข้อมูลปี 2569... ✅ สำเร็จ (58 แถว)

🎉 คลีนน้ำมันเสร็จเรียบร้อย! บันทึกไปที่: ../../data/processed\oil_wti_all_years_cleaned.csv


In [26]:
df_oil = pd.read_csv('../../data/processed/oil_wti_all_years_cleaned.csv')
df_oil

,Date,Oil_WTI_Close,Year_Scraped
0,2021-01-04,47.619999,2564
1,2021-01-05,49.930000,2564
2,2021-01-06,50.630001,2564
3,2021-01-07,50.830002,2564
4,2021-01-08,52.240002,2564
...,...,...,...
1307,2026-03-20,98.320000,2569
1308,2026-03-23,88.129997,2569
1309,2026-03-24,92.349998,2569
1310,2026-03-25,90.320000,2569


## **new_raw**

### **data Understanding**

In [ ]:
df = pd.read_csv('../../data/raw/new_raw/sp500_headlines_2008_2024.csv')
df['Date'] = pd.to_datetime(df['Date'])
df

,Title,Date,CP
0,"JPMorgan Predicts 2008 Will Be ""Nothing But Net""",2008-01-02,1447.16
1,Dow Tallies Biggest First-session-of-year Poin...,2008-01-02,1447.16
2,2008 predictions for the S&P 500,2008-01-02,1447.16
3,"U.S. Stocks Higher After Economic Data, Monsan...",2008-01-03,1447.16
4,U.S. Stocks Climb As Hopes Increase For More F...,2008-01-07,1416.18
...,...,...,...
19122,REITs vs. Stocks: What Does the Data Say?,2024-03-04,5130.95
19123,"Nasdaq Index, Dow Jones, S&P 500 News: Futures...",2024-03-04,5130.95
19124,"Nasdaq 100, Dow Jones, S&P 500 News: Cautious ...",2024-03-04,5130.95
19125,"Bank of America boosts S&P 500 target to 5,400...",2024-03-04,5130.95


### **Data Preparation**

In [ ]:
# 1. อ่านไฟล์ข่าว S&P 500
df = pd.read_csv('../../data/raw/new_raw/sp500_headlines_2008_2024.csv')
df['Date'] = pd.to_datetime(df['Date'])

# 2. กำหนด Keyword ที่บ่งบอกถึงภาวะสงครามและความขัดแย้ง
war_keywords = [
    'Russia', 'Ukraine', 'Invasion', 'War', 'Military', 'Conflict',
    'Gaza', 'Israel', 'Middle East', 'Attack', 'Tension', 'Sanction',
    'Geopolitical', 'Missile'
]

# 3. กรองข่าวที่มีคำเหล่านี้ (Case-insensitive) และอยู่ในช่วงปีที่กำหนด
# เน้นช่วงปี 2021 เป็นต้นไปที่มีสงครามรัสเซีย-ยูเครน และอิสราเอล
df_war = df[
    (df['Date'] >= '2021-01-01') &
    (df['Title'].str.contains('|'.join(war_keywords), case=False, na=False))
]

# 4. กำหนด Path ปลายทางและสร้าง Folder หากยังไม่มี
output_dir = '../../data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'war_related_news_2021_2024.csv')

# Save เป็นไฟล์ใหม่ไปยัง Path ที่กำหนด
df_war.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ บันทึกไฟล์สำเร็จที่: {output_path}")
print(f"สกัดข่าวที่เกี่ยวกับสงครามได้ทั้งหมด: {len(df_war)} ข่าว")

✅ บันทึกไฟล์สำเร็จที่: ../../data/processed\war_related_news_2021_2024.csv
สกัดข่าวที่เกี่ยวกับสงครามได้ทั้งหมด: 367 ข่าว


In [22]:
df_new = pd.read_csv('../../data/processed/war_related_news_2021_2024.csv')
df_new

,Title,Date,CP
0,What Warren Buffett's losing battle against S&...,2021-01-08,3824.68
1,The Top 10 Warren Buffett Stocks,2021-01-08,3824.68
2,Goldman Sachs warns of a dangerous bubble in t...,2021-01-25,3855.36
3,Stock Market Crash Warning,2021-02-18,3913.97
4,Small-cap stocks jump ahead of larger peers as...,2021-02-23,3881.37
...,...,...,...
362,Berkshire Hathaway Stock Rises To Record High—...,2024-02-26,5069.53
363,S&P 500 Gains and Losses Today: Index Slips Af...,2024-02-28,5069.76
364,"Nasdaq 100, Dow Jones, S&P 500 News: Upward Mo...",2024-02-29,5096.27
365,"Nasdaq 100, Dow Jones, S&P 500 News: Tech Rall...",2024-03-01,5137.08
